# Raw Dataset Processor

Pipeline monitoring untuk mengubah raw dataset `datasets/url_discovery/*.json` menjadi kandidat dataset dengan schema seperti `datasets/v1_shs_datasets.csv`. Notebook ini tidak menyimpan output secara otomatis.

## 1. Setup

In [100]:
from pathlib import Path
import sys

import polars as pl
from IPython.display import display


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "config.py").exists() and (candidate / "services").is_dir():
            return candidate
    raise FileNotFoundError("Root proyek tidak ditemukan")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from services.dataset_service import DatasetService

pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_cell_alignment("LEFT")
pl.Config.set_fmt_str_lengths(250)
pl.Config.set_tbl_width_chars(180)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)

dataset_service = DatasetService()
RAW_FOLDER = config.DATASETS / "url_discovery"
RESEARCH_CONFIG_PATH = PROJECT_ROOT / "research_config.json"
REFERENCE_DATASET_PATH = config.DATASETS / "reference_datasets.csv"

print(f"Raw folder: {RAW_FOLDER}")
print(f"Research config: {RESEARCH_CONFIG_PATH}")
print(f"Reference schema: {REFERENCE_DATASET_PATH}")

Raw folder: E:\School\tugas-akhir\project\datasets\url_discovery
Research config: E:\School\tugas-akhir\project\research_config.json
Reference schema: E:\School\tugas-akhir\project\datasets\reference_datasets.csv


## 2. Load Config dan Raw Discovery

In [101]:
research_config = dataset_service.load_research_config(RESEARCH_CONFIG_PATH)
meta_df = dataset_service.load_url_discovery_meta(RAW_FOLDER)
queries_df = dataset_service.load_url_discovery_queries(RAW_FOLDER)
raw_records_df = dataset_service.load_url_discovery_records(RAW_FOLDER)

RECORDS_FILTER_CONDITIONS = [
    # pl.col("content_status") == "success",
    # pl.col("content_text").fill_null("").str.strip_chars().str.len_chars() > 0,
    # ~pl.col("domain").str.contains("instagram.com|tiktok.com|facebook.com"),
]


def apply_filter_conditions(df: pl.DataFrame, conditions: list) -> pl.DataFrame:
    if not conditions:
        return df
    expression = conditions[0]
    for condition in conditions[1:]:
        expression = expression & condition
    return df.filter(expression)


records_df = apply_filter_conditions(raw_records_df, RECORDS_FILTER_CONDITIONS)

print("Search label:", research_config.get("search_label"))
print("Primary location:", research_config.get("primary_location"))
print(f"File metadata: {meta_df.height:,}")
print(f"Query rows: {queries_df.height:,}")
print(f"Raw records unik: {raw_records_df.height:,}")
print(f"Records setelah filter manual: {records_df.height:,}")

display(meta_df)

Search label: SHS/PLTS Kalimantan Barat
Primary location: Kalimantan Barat
File metadata: 8
Query rows: 2,067
Raw records unik: 1,364
Records setelah filter manual: 1,364


source_file,built_at,search_label,source_files,record_count,content_count,query_count
"""dataset_20260627T094855Z.json""","""2026-06-27T09:48:55.240893+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",214,214,335
"""dataset_20260627T101253Z.json""","""2026-06-27T10:12:53.267116+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",203,203,268
"""dataset_20260627T135259Z.json""","""2026-06-27T13:52:59.725144+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",88,88,197
"""dataset_20260628T033631Z.json""","""2026-06-28T03:36:31.219965+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",202,202,381
"""dataset_20260628T104512Z.json""","""2026-06-28T10:45:12.810124+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",136,136,369
"""dataset_20260628T113525Z.json""","""2026-06-28T11:35:25.172805+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",12,12,28
"""dataset_20260628T173013Z.json""","""2026-06-28T17:30:13.810625+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",250,250,267
"""dataset_20260630T005918Z.json""","""2026-06-30T00:59:18.792990+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",259,259,490


## 3. Monitoring Kualitas Raw Dataset

In [102]:
status_distribution = (
    records_df.with_columns(pl.col("content_status").fill_null("missing"))
    .group_by("content_status")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

domain_distribution = (
    records_df.with_columns(pl.col("domain").fill_null("unknown"))
    .group_by("domain")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

query_group_distribution = (
    records_df.with_columns(pl.col("query_group").fill_null("unknown"))
    .group_by("query_group")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

text_length_summary = records_df.select(
    pl.len().alias("total_records"),
    (
        pl.col("content_text")
        .fill_null("")
        .str.strip_chars()
        .str.len_chars()
        > 0
    ).sum().alias("records_with_text"),
    pl.col("content_text_length").min().alias("min_text_length"),
    pl.col("content_text_length").mean().round(2).alias("avg_text_length"),
    pl.col("content_text_length").max().alias("max_text_length"),
)

display(status_distribution)
display(domain_distribution)
display(query_group_distribution)
display(text_length_summary)

content_status,jumlah
"""success""",1067
"""too_short""",200
"""pdf_skipped""",47
"""failed""",41
"""pending""",9


domain,jumlah
"""www.instagram.com""",530
"""www.tiktok.com""",190
"""www.facebook.com""",56
"""pontianakpost.jawapos.com""",32
"""rri.co.id""",28
"""kalbar.antaranews.com""",25
"""www.researchgate.net""",12
"""id.scribd.com""",11
"""krisnamandiri.com""",10
"""www.krisnamandiriutama.com""",9


query_group,jumlah
"""core:electricity_access""",232
"""core:grid_hybrid""",115
"""core:household_solar""",103
"""issue:benefit""",93
"""issue:problem""",84
"""social""",75
"""core:technical""",68
"""core:government_program""",67
"""issue:access""",62
"""issue:environment""",55


total_records,records_with_text,min_text_length,avg_text_length,max_text_length
1364,1099,0,3257.82,33760


## 4. Bangun Kandidat Schema v1

In [ ]:
candidate_df = dataset_service.build_v1_candidate_rows(
    records_df=records_df,
    research_config=research_config,
)

CANDIDATE_FILTER_CONDITIONS = [
    # pl.col("dataset_tier") == "C_holdout_excluded",
    # pl.col("source_type") == "social_media",
    # pl.col("source_url").str.to_lowercase().str.contains("facebook.com"),
    # ~pl.col("text").str.to_lowercase().str.contains("penayangan"),
    # pl.col("aspect").is_in(["experience", "benefit", "access"]),
]

# A_candidate_core
# B_review_queue
# C_holdout_excluded

# "social_media"
# "online_news"
# "academic_repository"
# "government_official"

filtered_candidate_df = apply_filter_conditions(candidate_df, CANDIDATE_FILTER_CONDITIONS)

reference_df = dataset_service.load(REFERENCE_DATASET_PATH)
expected_columns = list(dataset_service.v1_dataset_columns())
missing_candidate_columns = [column for column in expected_columns if column not in filtered_candidate_df.columns]
extra_candidate_columns = [column for column in filtered_candidate_df.columns if column not in expected_columns]

print(f"Candidate rows sebelum filter kandidat: {candidate_df.height:,}")
print(f"Candidate rows setelah filter kandidat: {filtered_candidate_df.height:,}")
print("Kolom wajib hilang:", missing_candidate_columns)
print("Kolom monitoring tambahan:", extra_candidate_columns)
print("Urutan kolom utama sama:", filtered_candidate_df.columns[: len(expected_columns)] == expected_columns)

display(filtered_candidate_df.select(expected_columns).head(20))

Candidate rows sebelum filter kandidat: 1,364
Candidate rows setelah filter kandidat: 203
Kolom wajib hilang: []
Kolom monitoring tambahan: ['raw_source_file', 'raw_domain', 'content_status', 'query_group', 'query', 'raw_title', 'raw_text_length']
Urutan kolom utama sama: True


text_id,text,subjectivity_type,speaker_type,public_opinion_scope,corpus_role,aspect,location,sentiment_label,label_status,source_id,source_type,source_url,dataset_tier,inclusion_status,verification_status,evidence_support_score,parent_text_id,decision_note
"""RAW-0006""","""TikTok - Make Your Day Listrik Padam Sumatera: Solusi PLTS dan Stok Cadangan ... TikTok · evanstoryy 7,2 rb+ penayangan @evanstoryyy 7643019382251080967""","""contextual_source""","""public_user""","""contextual_reference""","""excluded""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0006""","""social_media""","""https://www.tiktok.com/@evanstoryyy/video/7643019382251080967""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.55,"""""","""content_not_success; social_domain_review"""
"""RAW-0008""","""TikTok - Make Your Day Teknologi CSP China di Gurun Gobi: Pembangkit Listrik 24 ... TikTok · bacaaja.co 3,8 rb+ penayangan @bacaaja.co 7651132761448647956""","""contextual_source""","""public_user""","""contextual_reference""","""excluded""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0008""","""social_media""","""https://www.tiktok.com/@bacaaja.co/video/7651132761448647956""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.55,"""""","""content_not_success; social_domain_review"""
"""RAW-0012""","""TikTok - Make Your Day Program Listrik Desa Perluas Akses Listrik ke Dusun Terpencil TikTok · Dunia Punya Cerita 92 rb+ penayangan · 4 hari yang lalu @duniapunyacerita 7654119865329978644""","""contextual_source""","""public_user""","""contextual_reference""","""excluded""","""electricity_access""","""""","""""","""unlabeled""","""RAW-SRC-0012""","""social_media""","""https://www.tiktok.com/@duniapunyacerita_/video/7654119865329978644""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.35,"""""","""content_not_success; social_domain_review"""
"""RAW-0017""","""TikTok - Make Your Day Usulan Pembangunan Infrastruktur Listrik di Ketapang TikTok · Alexander Wilyo - AW 15,7 rb+ penayangan · 6 bulan yang lalu @alexanderwilyo 7579183210689482005""","""contextual_source""","""public_user""","""contextual_reference""","""excluded""","""general_shs""","""Ketapang""","""""","""unlabeled""","""RAW-SRC-0017""","""social_media""","""https://www.tiktok.com/@alexanderwilyo/video/7579183210689482005""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.55,"""""","""content_not_success; social_domain_review"""
"""RAW-0043""","""TikTok - Make Your Day Pasang PLTS untuk Tetangga: Solusi Energik TikTok · Bapakcanggih 232 rb+ penayangan · 3 bulan yang lalu @ilhambachtiarr 7614778147426487572""","""contextual_source""","""public_user""","""contextual_reference""","""excluded""","""general_shs""","""Sintang""","""""","""unlabeled""","""RAW-SRC-0043""","""social_media""","""https://www.tiktok.com/@ilhambachtiarr/video/7614778147426487572""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.55,"""""","""content_not_success; social_domain_review"""
"""RAW-0059""","""TikTok - Make Your Day Pemadaman Listrik Massal di Sumatra: Update Terbaru TikTok · OfficialiNews 247,8 rb+ penayangan @officialinews 7642928925978676501""","""contextual_source""","""public_user""","""contextual_reference""","""excluded""","""electricity_access""","""""","""""","""unlabeled""","""RAW-SRC-0059""","""social_media""","""https://www.tiktok.com/@officialinews/photo/7642928925978676501?image_index=1""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.35,"""""","""content_not_success; social_domain_review"""
"""RAW-0068""","""TikTok - Make Your Day Bupati Kapuas Pimpin Rapat Elektrifikasi Desa Barunang TikTok · Pemkab Kapuas 8,2 rb+ penayangan · 1 minggu yang lalu @pemkabkapuas 7652875267463023890""","""contextual_source""","""public_user""","""contextua

## 5. Monitoring Kandidat

In [104]:
tier_distribution = (
    filtered_candidate_df.group_by("dataset_tier")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

inclusion_distribution = (
    filtered_candidate_df.group_by("inclusion_status")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

source_type_distribution = (
    filtered_candidate_df.group_by("source_type")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

aspect_distribution = (
    filtered_candidate_df.group_by("aspect")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)


location_distribution = (
    filtered_candidate_df.group_by("location")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

display(tier_distribution)
display(inclusion_distribution)
display(source_type_distribution)
display(aspect_distribution)
display(location_distribution)

dataset_tier,jumlah
"""C_holdout_excluded""",203


inclusion_status,jumlah
"""held_out_not_for_sentiment_core""",203


source_type,jumlah
"""social_media""",203


aspect,jumlah
"""general_shs""",143
"""electricity_access""",23
"""grid_hybrid""",9
"""experience""",8
"""environment""",6
"""household_solar""",5
"""benefit""",4
"""access""",3
"""village_solar""",2


location,jumlah
"""""",24
"""Kalimantan Barat""",23
"""Kubu Raya""",22
"""Singkawang""",15
"""Pontianak""",15
"""Sintang""",14
"""Landak""",13
"""Sambas""",13
"""Ketapang""",13
"""Kapuas Hulu""",10


## 6. Review Kandidat Prioritas

In [105]:
review_columns = [
    "text_id",
    "dataset_tier",
    "inclusion_status",
    "evidence_support_score",
    "aspect",
    "location",
    "source_type",
    "source_url",
    "text",
    "decision_note",
    "query_group",
    "query",
]

priority_df = (
    filtered_candidate_df
    .filter(pl.col("dataset_tier").is_in(["A_candidate_core", "B_review_queue"]))
    .sort(["dataset_tier", "evidence_support_score"], descending=[False, True])
    .select(review_columns)
)

display(priority_df.head(50))

text_id,dataset_tier,inclusion_status,evidence_support_score,aspect,location,source_type,source_url,text,decision_note,query_group,query


## 7. Export Manual Opsional

In [106]:
EXPORT_CSV = False
EXPORT_PATH = config.OUTPUTS / "datasets" / "candidate_datasets.csv"

if EXPORT_CSV:
    EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    filtered_candidate_df.write_csv(EXPORT_PATH)
    print(f"Candidate dataset terfilter disimpan ke: {EXPORT_PATH}")
else:
    print("Export dimatikan. Ubah EXPORT_CSV = True jika ingin menyimpan kandidat terfilter secara manual.")

Export dimatikan. Ubah EXPORT_CSV = True jika ingin menyimpan kandidat terfilter secara manual.
